# Feature Deletion Sensitivity

Measures model sensitivity to deletion of individual semantic features. Bridges interpretability from granular components to human-understandable concepts through systematic deletion analysis.

**References:**
- Bricken et al. (2023) "Towards Monosemanticity"
- Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"

## Why This Matters

Feature deletion sensitivity measures how much removing individual features (neurons, directions, or SAE features) impacts predictions. Deletion cascades and interaction tests reveal which features are independently important vs. which only matter in combination.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_deletion_sensitivity import (
    feature_importance_ranking,
    feature_deletion_cascade,
    feature_interaction_effects,
    minimal_sufficient_features,
    feature_redundancy_clusters,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
def metric(logits): return float(logits[-1, 0])

# Create random feature directions (in practice, from SAE)
rng = np.random.RandomState(42)
features = rng.randn(8, 32).astype(np.float32)
features /= np.linalg.norm(features, axis=1, keepdims=True)
print(f"Model: {cfg.n_layers} layers, {len(features)} features")

In [ ]:
# 1. Feature importance ranking
ranking = feature_importance_ranking(model, tokens, features, metric)
print(f"Top feature: {ranking['top_feature']}")
print(f"Importance scores: {[f'{s:.4f}' for s in ranking['importance_scores']]}")
print(f"Ranking: {ranking['ranking']}")

In [ ]:
# 2. Deletion cascade
cascade = feature_deletion_cascade(model, tokens, features, metric)
print(f"Deletion order: {cascade['deletion_order']}")
print(f"Metric at half: {cascade['metric_at_half']:.4f}")
print(f"Cumulative metrics: {[f'{m:.4f}' for m in cascade['cumulative_metrics']]}")

In [ ]:
# 3. Feature interactions
interactions = feature_interaction_effects(
    model, tokens, features[:4], metric, feature_pairs=[(0,1), (0,2), (1,2)]
)
print(f"Interactions: {interactions['interaction_matrix']}")
print(f"Strongest: {interactions['strongest_interaction']}")
print(f"Mean: {interactions['mean_interaction']:.4f}")

In [ ]:
# 4. Redundancy clusters
clusters = feature_redundancy_clusters(model, features, n_clusters=3)
print(f"Cluster assignments: {clusters['cluster_assignments']}")
print(f"Cluster sizes: {clusters['cluster_sizes']}")
print(f"Within-cluster sim: {[f'{s:.3f}' for s in clusters['within_cluster_similarity']]}")
print(f"Between-cluster sim: {clusters['between_cluster_similarity']:.3f}")